# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:** Kunal Ranjan  
**`Roll Number`:**  U20230022
**`GitHub Branch`:** Kunal_U20230022

# Imports and Setup

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from rlcmab_sampler import sampler


# Load Datasets

In [33]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [34]:
print("Missing values in train_users:\n", train_users.isnull().sum())
print("\nMissing values in test_users:\n", test_users.isnull().sum())

#Fill missing age with median from training set
age_median = train_users['age'].median()
train_users['age'] = train_users['age'].fillna(age_median)
test_users['age'] = test_users['age'].fillna(age_median)

#Encoding
le = LabelEncoder()
train_users['label_encoded'] = le.fit_transform(train_users['label'])
print("\nLabel mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

feature_cols = ['age', 'income', 'clicks', 'purchase_amount']

X_train = train_users[feature_cols].values
y_train = train_users['label_encoded'].values

X_test = test_users[feature_cols].values


print(f"\nTraining set: X={X_train.shape}, y={y_train.shape}")
print(f"Test set:     X={X_test.shape} (no labels)")
print(f"\nClass distribution (train):\n{train_users['label'].value_counts()}")

Missing values in train_users:
 user_id                          0
age                            698
income                           0
clicks                           0
purchase_amount                  0
session_duration                 0
content_variety                  0
engagement_score                 0
num_transactions                 0
avg_monthly_spend                0
avg_cart_value                   0
browsing_depth                   0
revisit_rate                     0
scroll_activity                  0
time_on_site                     0
interaction_count                0
preferred_price_range            0
discount_usage_rate              0
wishlist_size                    0
product_views                    0
repeat_purchase_gap (days)       0
churn_risk_score                 0
loyalty_index                    0
screen_brightness                0
battery_percentage               0
cart_abandonment_count           0
browser_version                  0
background_app_count   

## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Convert boolean 'subscriber' to int
train_users['subscriber'] = train_users['subscriber'].astype(int)
test_users['subscriber'] = test_users['subscriber'].astype(int)

# --- One-hot encode categorical columns instead of dropping them ---
categorical_cols = ['browser_version', 'region_code']

train_encoded = pd.get_dummies(train_users, columns=categorical_cols, dtype=int)
test_encoded  = pd.get_dummies(test_users,  columns=categorical_cols, dtype=int)

# Align columns: keep only columns present in both sets (handles unseen categories)
common_cols = train_encoded.columns.intersection(test_encoded.columns)
train_encoded = train_encoded[common_cols]
test_encoded  = test_encoded[common_cols]

# Drop non-feature columns
drop_cols = ['user_id', 'label', 'label_encoded']
feature_cols_v2 = [c for c in train_encoded.columns if c not in drop_cols]

print(f"Total features used: {len(feature_cols_v2)}")
print(f"  (includes {sum(1 for c in feature_cols_v2 if c.startswith('browser_version_'))} browser_version dummies, "
      f"{sum(1 for c in feature_cols_v2 if c.startswith('region_code_'))} region_code dummies)")

X_full = train_encoded[feature_cols_v2].values
X_test_full = test_encoded[feature_cols_v2].values

X_tr, X_val, y_tr, y_val = train_test_split(X_full, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Scaling
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)

# Decision Tree
dt_clf = DecisionTreeClassifier(random_state=42, max_depth=15)
dt_clf.fit(X_tr_scaled, y_tr)
dt_acc = accuracy_score(y_val, dt_clf.predict(X_val_scaled))

# Logistic Regression
lr_clf = LogisticRegression(max_iter=3000, random_state=42, C=1.0, solver='lbfgs')
lr_clf.fit(X_tr_scaled, y_tr)
lr_acc = accuracy_score(y_val, lr_clf.predict(X_val_scaled))

# Random Forest
rf_clf = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42)
rf_clf.fit(X_tr_scaled, y_tr)
rf_acc = accuracy_score(y_val, rf_clf.predict(X_val_scaled))

# Comparison
print("\nModel Comparison (on 20% validation split)")
print(f"Decision Tree       accuracy: {dt_acc:.4f}")
print(f"Logistic Reg.       accuracy: {lr_acc:.4f}")
print(f"Random Forest       accuracy: {rf_acc:.4f}")

# Best pick
best_name, best_acc, best_clf, uses_scaler = max(
    [("Decision Tree", dt_acc, dt_clf, False),
     ("Logistic Regression", lr_acc, lr_clf, True),
     ("Random Forest", rf_acc, rf_clf, False)],
    key=lambda x: x[1]
)
print(f"\nBest model: {best_name} ({best_acc:.4f})")

# Refit best model on full training data for downstream use
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_full)
X_test_scaled = scaler.transform(X_test_full)
best_clf.fit(X_train_scaled, y_train)

# Storing the best classifier for later use
context_classifier = best_clf

def predict_user_context(features):
    """Predict user category (0=user1, 1=user2, 2=user3) from raw features."""
    features = np.array(features).reshape(1, -1)
    if uses_scaler:
        features = scaler.transform(features)
    return context_classifier.predict(features)[0]

Total features used: 29

Model Comparison (on 20% validation split)
Decision Tree       accuracy: 0.7650
Logistic Reg.       accuracy: 0.7625
Random Forest       accuracy: 0.8450

Best model: Random Forest (0.8450)


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
